# Predicted Latency-Based Routing

Most routing strategies rely on proxy signals like queue depth or cache
affinity to estimate which replica will serve a request fastest. **Predicted
latency routing** takes a more direct approach: it trains a lightweight
XGBoost model online using live traffic data, then uses it to predict the
actual TTFT and time-per-output-token (TPOT) for each replica given the
current request.

This approach also enables **SLO-aware admission control**. Clients can
attach latency SLO headers to their requests, and the router will only
route to replicas where the predicted latency meets the SLO.

By the end of this notebook, you will:
- Deploy the predicted latency routing guide
- Configure the latency predictor plugin and SLO admitter
- Send requests with SLO headers and observe routing decisions
- Compare TTFT and ITL against heuristic routing
- Observe how the predictor improves over time

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"

print("Namespace:", os.environ["NAMESPACE"])
print("Model:", os.environ["MODEL_NAME"])

## How Predicted Latency Routing Works

The system has three components:

### 1. Data Collection
The EPP records features and outcomes for every completed request:
- **Features**: prompt length, max_tokens, current queue depth on the
  selected replica, KV-cache utilization, number of active requests
- **Outcomes**: observed TTFT, observed TPOT

### 2. Online Training
An XGBoost model is retrained periodically (every few minutes) on the
most recent data. This allows the model to adapt to changing conditions:
workload shifts, replica scaling events, model warmup, etc.

### 3. Prediction and Admission
When a new request arrives:
1. The predictor estimates TTFT and TPOT for each healthy replica.
2. If the request includes SLO headers, the SLO admitter filters out
   replicas where the predicted latency exceeds the SLO.
3. The remaining replicas are scored, and the one with the lowest
   predicted latency is selected.

```
Request + SLO headers
        |
        v
+----------------+
| Latency        |  Predict TTFT/TPOT for each replica
| Predictor      |
+-------+--------+
        |
        v
+----------------+
| SLO Admitter   |  Filter replicas that would violate SLO
+-------+--------+
        |
        v
  Select replica with lowest predicted latency
```

In [ ]:
# Deploy the predicted latency routing guide

# Model server (same as optimized baseline, but with additional metrics)
!kubectl apply -k llm-d-production-stack/guides/predicted-latency-routing/model-server/ \
    -n $NAMESPACE

print("Model server deployed.")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
print("Model server pods are ready.")

In [ ]:
# Deploy the router with latency predictor and SLO admitter

!kubectl apply -k llm-d-production-stack/guides/predicted-latency-routing/router/ \
    -n $NAMESPACE

print("Router deployed with latency predictor plugin.")
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s
print("EPP pods are ready.")

# Show the EPP config for the latency predictor
print("\n--- Latency Predictor Configuration ---")
!kubectl get configmap epp-config -n $NAMESPACE -o yaml | grep -A 20 'latency'

## Sending Requests with SLO Headers

The latency SLO admitter reads two optional headers from each request:

- **`x-slo-ttft-ms`**: Maximum acceptable time-to-first-token in milliseconds.
  The admitter rejects replicas where predicted TTFT exceeds this value.

- **`x-slo-tpot-ms`**: Maximum acceptable time-per-output-token in milliseconds.
  The admitter rejects replicas where predicted TPOT exceeds this value.

If no SLO headers are present, the request is routed purely by predicted
latency (lowest wins) without admission filtering.

In [ ]:
# Send requests with SLO headers
import subprocess
import json
import time

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

def send_with_slo(prompt, ttft_slo_ms=None, tpot_slo_ms=None, max_tokens=100):
    """Send a request with optional SLO headers."""
    headers = ["Content-Type: application/json"]
    if ttft_slo_ms is not None:
        headers.append(f"x-slo-ttft-ms: {ttft_slo_ms}")
    if tpot_slo_ms is not None:
        headers.append(f"x-slo-tpot-ms: {tpot_slo_ms}")

    payload = json.dumps({
        "model": "Qwen/Qwen3-32B",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
    })

    cmd = ["curl", "-s", "-w", "\n%{time_starttransfer} %{time_total}"]
    for h in headers:
        cmd.extend(["-H", h])
    cmd.extend([f"http://{GATEWAY_IP}:8080/v1/chat/completions", "-d", payload])

    result = subprocess.run(cmd, capture_output=True, text=True)
    lines = result.stdout.strip().split("\n")
    timing = lines[-1].split() if len(lines) > 1 else ["0", "0"]
    return {
        "ttft": float(timing[0]),
        "total": float(timing[1]),
        "status": "ok" if '"error"' not in result.stdout else "rejected",
    }

# First, warm up the predictor with some requests (no SLO)
print("Phase 1: Warming up the latency predictor (20 requests, no SLO)...")
for i in range(20):
    send_with_slo(f"Write a haiku about the number {i}.")
print("  Warmup complete. Predictor has training data.")

print()

# Now send requests with SLO headers
print("Phase 2: Sending requests with SLO headers")
print()

# Tight SLO - only fast replicas will be selected
print("Tight SLO (TTFT < 200ms, TPOT < 30ms):")
for i in range(5):
    r = send_with_slo("What is 2+2?", ttft_slo_ms=200, tpot_slo_ms=30)
    print(f"  Request {i+1}: TTFT={r['ttft']*1000:.0f}ms, "
          f"Total={r['total']*1000:.0f}ms, Status={r['status']}")

print()

# Relaxed SLO - any replica should be acceptable
print("Relaxed SLO (TTFT < 2000ms, TPOT < 100ms):")
for i in range(5):
    r = send_with_slo(
        "Explain the theory of relativity in detail.",
        ttft_slo_ms=2000, tpot_slo_ms=100, max_tokens=200
    )
    print(f"  Request {i+1}: TTFT={r['ttft']*1000:.0f}ms, "
          f"Total={r['total']*1000:.0f}ms, Status={r['status']}")

In [ ]:
# Generate mixed workloads (short and long prompts) to test routing quality
import threading

short_prompts = [
    "What is 2+2?",
    "Name three colors.",
    "What day comes after Monday?",
    "Is water wet?",
    "What is the capital of France?",
]

long_prompts = [
    "Write a detailed essay about the history of computing, covering the "
    "invention of the transistor, the development of integrated circuits, "
    "the rise of personal computers, and the emergence of cloud computing. "
    "Include specific dates and key figures in each era.",

    "Explain the complete process of how a neural network learns, starting "
    "from random weight initialization, through forward propagation, loss "
    "calculation, backpropagation, and gradient descent. Include mathematical "
    "formulas where appropriate and discuss common challenges like vanishing "
    "gradients and overfitting.",

    "Describe the architecture of a modern microprocessor in detail. Cover "
    "the fetch-decode-execute cycle, pipelining, branch prediction, out-of-order "
    "execution, cache hierarchy, and memory management. Compare CISC and RISC "
    "design philosophies.",
]

results = {"short": [], "long": []}

def benchmark_prompt(prompt, category, max_tokens):
    r = send_with_slo(prompt, max_tokens=max_tokens)
    results[category].append(r)

print("Running mixed workload benchmark...")
print("  Short prompts: 5 (max_tokens=50)")
print("  Long prompts: 3 (max_tokens=300)")
print()

threads = []
for p in short_prompts:
    t = threading.Thread(target=benchmark_prompt, args=(p, "short", 50))
    threads.append(t)
for p in long_prompts:
    t = threading.Thread(target=benchmark_prompt, args=(p, "long", 300))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

# Report results
print("=== Short Prompts ===")
short_ttfts = [r["ttft"] * 1000 for r in results["short"]]
print(f"  Avg TTFT: {sum(short_ttfts)/len(short_ttfts):.0f} ms")
print(f"  Min TTFT: {min(short_ttfts):.0f} ms")
print(f"  Max TTFT: {max(short_ttfts):.0f} ms")

print()
print("=== Long Prompts ===")
long_ttfts = [r["ttft"] * 1000 for r in results["long"]]
print(f"  Avg TTFT: {sum(long_ttfts)/len(long_ttfts):.0f} ms")
print(f"  Min TTFT: {min(long_ttfts):.0f} ms")
print(f"  Max TTFT: {max(long_ttfts):.0f} ms")

print()
print("The latency predictor should route short prompts to less-loaded replicas")
print("and long prompts to replicas with more available KV-cache space.")

In [ ]:
# Compare against heuristic routing
# The EPP logs include the routing decision reason, showing which
# scorer contributed to the selection

print("=== Routing Decision Comparison ===")
print()
print("With predicted latency routing, the EPP logs show:")
print()

# Check recent EPP logs for routing decisions
!kubectl logs -n llm-d -l app=epp --tail=20 | grep -i 'routing\|selected\|predict' | head -10

print()
print("Expected improvements over heuristic routing:")
print("  TTFT:  10-30% lower (predictor accounts for actual model load)")
print("  ITL:   5-15% lower (better replica selection under contention)")
print("  Tail:  20-40% lower p99 (predictor avoids overloaded replicas)")
print()
print("The improvement grows with more replicas, because the predictor")
print("can differentiate between replicas that look similar to heuristics")
print("but have different real-world latency characteristics.")

In [ ]:
# Show how the predictor adapts over time
# The predictor retrains periodically and its accuracy improves
# as it collects more training data

print("=== Predictor Training Status ===")
print()

# Check predictor metrics
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=epp_predictor_training_samples_total{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    print(f'  Training samples collected: {r["value"][1]}')
" 2>/dev/null || echo "  (metrics not yet available)"

print()

!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=epp_predictor_mae{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    target = r['metric'].get('target', 'unknown')
    mae = float(r['value'][1]) * 1000
    print(f'  {target} prediction MAE: {mae:.1f} ms')
" 2>/dev/null || echo "  (metrics not yet available)"

print()
print("The predictor's mean absolute error (MAE) should decrease over time")
print("as it collects more training data. Typical convergence:")
print("  0-100 samples:   MAE ~200ms (cold start, predictions are rough)")
print("  100-1000:        MAE ~50ms (model learns workload patterns)")
print("  1000+:           MAE ~20ms (model has seen enough variety)")
print()
print("The model retrains every 2 minutes by default. You can adjust")
print("the retraining interval in the EPP config.")

## Summary

Predicted latency routing replaces heuristic proxy signals with a
data-driven model that learns from live traffic.

### Key Components

- **Latency Predictor**: XGBoost model trained online on request features
  and observed latencies. Retrains every 2 minutes.
- **SLO Admitter**: Filters out replicas where predicted latency would
  violate the client's SLO (set via `x-slo-ttft-ms` and `x-slo-tpot-ms`
  headers).

### When to Use

- Workloads with strict latency SLOs that vary per request
- Heterogeneous clusters where replicas have different performance
  characteristics (e.g., mixed GPU types)
- High-scale deployments where heuristic routing leaves performance
  on the table

### Limitations

- Cold start: The predictor needs ~100 requests to start making useful
  predictions. During cold start, it falls back to heuristic routing.
- Workload shifts: If the workload changes dramatically, the predictor
  takes a few retraining cycles to adapt.

### Next Steps

- **Router Dashboard**: Set up Grafana to visualize predictor accuracy
  and SLO compliance over time.
- **Flow Control**: Combine latency prediction with priority bands for
  multi-tenant SLO management.